# 03 · One synchronised map per attribute (ComponentMatrixViewer)

Every neuron holds a multi-attribute weight vector. The `ComponentMatrixViewer` shows all attributes at once, one
map each, in a grid. In each map the triangle between three neighbouring neurons is coloured by how much *that*
attribute changes between them (mean |w_a[k] − w_b[k]|), so you can see which attributes form which cluster
borders. The first map is the Euclidean distance over all attributes, for reference.

With live figures, drag any map and all of them rotate together; click to mark a neuron on every map (its value of
each attribute appears in the titles); press **v** to switch between |difference| and value and **c** for one
colour scale.

Script version: [`examples/03_component_matrix.py`](../03_component_matrix.py) · Needs: numpy, mtgeodesicdome, matplotlib, ipywidgets
(optional: `ipympl` for live figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicsom importable from a checkout; helpers for the controls below

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import ipywidgets as widgets
import numpy as np

from mt.geodesicsom import datasets
from mt.geodesicsom.GeodesicSOM import GeodesicSOM
from mt.geodesicsom.gui import ComponentMatrixViewer

## 1. Data and training

In [ ]:
DATA = 'clusters'

x, y, names = datasets.load(DATA)
if DATA != 'colours':
    x = datasets.standardise(x)

som = GeodesicSOM(10, seed=3)
som.initialise(dataset=x)
som.train(x, epochs=30)
print(f'{som}: {len(x)} samples, QE {som.quantisation_error(x):.3f}, TE {som.topographic_error(x):.3f}')

## 2. Which attributes make the borders?

Before looking at the maps, a number per attribute: the mean change of that attribute between neighbouring
neurons (`face_component_difference`), and the share of the *strongest* borders (the top 10 % of triangles by
Euclidean distance) that the attribute contributes to.

In [ ]:
diff = som.face_component_difference()                 # (faces, attributes)
total = som.face_distance()                            # (faces,)
ridge = total >= np.quantile(total, 0.9)
share = diff[ridge].mean(axis=0) / diff[ridge].mean(axis=0).sum()

print(f'{"attribute":14s} {"mean change":>12s} {"share of top-10% borders":>26s}')
for k in np.argsort(-share):
    print(f'{names[k]:14s} {diff[:, k].mean():12.3f} {share[k]:26.1%}')

## 3. The matrix

> **Live or static?** With `ipympl` installed the viewers below are live: drag a sphere to rotate it, click a
> neuron to inspect it, and use the radio buttons, check boxes and keys on the figure itself. Without it they are
> images, and the controls under each viewer redraw them.

In [ ]:
matrix = ComponentMatrixViewer(som, names, mode='difference', data=x, labels=y)

### Controls

Colour by the attribute's difference between neighbours or by its value, use one colour scale for all maps,
select a neuron, and turn the spheres.

In [ ]:
nb_setup.controls(matrix, neuron=int(np.argmax(som.hits(x))))

## 4. One attribute on its own

`open_single(k)` opens a full `SOMViewer` of attribute *k* (a double-click on a map does the same), linked to
the matrix: it shares the rotation and the selected neuron.

In [ ]:
strongest = int(np.argmax(share))
with nb_setup.building():
    single = matrix.open_single(strongest)
print('opened', names[strongest])
nb_setup.show(single)

In [ ]:
nb_setup.controls(single, matrix)

### Try it: another data set

Press **Run Interact** to train a sphere on another sample and show its matrix.

In [ ]:
@widgets.interact_manual(data=widgets.Dropdown(options=[*datasets.samples(), 'colours'], value='wine'),
                         freq=widgets.IntSlider(value=8, min=2, max=16, description='frequency'),
                         mode=widgets.ToggleButtons(options=['difference', 'value']),
                         shared_scale=widgets.Checkbox(value=False, description='one colour scale'),
                         include_total=widgets.Checkbox(value=True, description="'all attributes' map"))
def train_matrix(data, freq, mode, shared_scale, include_total):
    x, y, names = datasets.load(data)
    if data != 'colours':
        x = datasets.standardise(x)
    som = GeodesicSOM(freq, seed=3)
    som.initialise(dataset=x)
    som.train(x, epochs=30)
    print(f'{som}: QE {som.quantisation_error(x):.3f}, TE {som.topographic_error(x):.3f}')
    with nb_setup.building():
        m = ComponentMatrixViewer(som, names, mode=mode, shared_scale=shared_scale,
                                  include_total=include_total, data=x, labels=y)
    nb_setup.show(m)
    nb_setup.controls(m)